In [19]:
%pip install huggingface_hub ddgs sentence-transformers transformers scikit-learn numpy pydantic --quiet

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import re
import json
import warnings
import numpy as np
import transformers
from huggingface_hub import InferenceClient
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline as hf_pipeline
from ddgs import DDGS

# Suppress noisy model-load warnings and progress bars
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

HF_API_KEY = os.environ.get("HF_API_KEY")
GEN_MODEL  = "deepseek-ai/DeepSeek-V3.2"
model_name = "all-MiniLM-L6-v2"

client = InferenceClient(api_key=HF_API_KEY)


def chat(messages: list, max_tokens: int = 1024, temperature: float = 0.1) -> str:
    """Call the generative model via HuggingFace Inference API."""
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content


def chat_json(user_prompt: str, template: str, max_tokens: int = 1024) -> dict:
    """Ask the model to fill in a JSON template and return parsed dict."""
    prompt = (
        f"{user_prompt}\n\n"
        "Complete the following JSON by replacing every placeholder value "
        "with real content. Output ONLY the completed JSON — no extra text:\n\n"
        f"{template}"
    )
    raw = chat(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.0,
    )
    return extract_json(raw)


def extract_json(text: str) -> dict:
    """Aggressively extract a JSON object from model output."""
    text = re.sub(r"```(?:json)?[^\n]*\n?", "", text).strip(" `\n")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    raise ValueError(f"No valid JSON found:\n{text[:400]}")


print(f"Model : {GEN_MODEL}")
print(f"Embeds: {model_name}")
print(f"API   : {'set' if HF_API_KEY else 'MISSING'}")

Model : deepseek-ai/DeepSeek-V3.2
Embeds: all-MiniLM-L6-v2
API   : MISSING


In [21]:
class ResearchPlan(BaseModel):
    tasks: list[str]        # specific web search queries (6-8)
    focus_areas: list[str]  # high-level themes to cover

class ResearchFinding(BaseModel):
    query: str
    content: str

class MLInsights(BaseModel):
    sentiment: dict          # {positive: float, negative: float, neutral: float}
    clusters: dict           # {theme_id: [chunk_previews]}
    entities: dict           # {ORG: [...], PER: [...]}
    top_findings: list[str]  # relevance-ranked chunks

class SWOTAnalysis(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    opportunities: list[str]
    threats: list[str]

class Analysis(BaseModel):
    swot: SWOTAnalysis
    key_metrics: list[str]
    competitive_landscape: str
    main_insights: list[str]

print("Pydantic models defined.")

Pydantic models defined.


In [22]:
class MLProcessingLayer:
    """Hybrid ML layer: local DL models enrich data before passing to the LLM."""

    def __init__(self):
        print("Loading local ML models...")
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self.sentiment_pipe = hf_pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            truncation=True, max_length=512
        )
        # TokenClassificationPipeline does not accept truncation=;
        # inputs are sliced to 512 chars manually before each call.
        self.ner_pipe = hf_pipeline(
            "ner",
            model="dslim/bert-base-NER",
            aggregation_strategy="simple",
        )
        print("ML models ready.")

    def process(self, query: str, findings: list) -> MLInsights:
        chunks = [f.content for f in findings]
        if not chunks:
            return MLInsights(sentiment={}, clusters={}, entities={}, top_findings=[])

        # 1. EMBEDDINGS
        # Sentence-BERT encodes each chunk into a 384-dim vector.
        # Similar sentences cluster together in this continuous vector space.
        embeddings = self.embedder.encode(chunks, show_progress_bar=False)  # (N, 384)
        query_emb  = self.embedder.encode([query])                          # (1, 384)

        # 2. COSINE SIMILARITY — relevance ranking
        # cos_sim(a,b) = dot(a,b) / (||a|| * ||b||)  in [-1, 1]
        # Magnitude-invariant: only direction (meaning) matters.
        sim_scores   = cosine_similarity(query_emb, embeddings)[0]
        ranked_idx   = np.argsort(sim_scores)[::-1]
        top_findings = [chunks[i] for i in ranked_idx[:5]]

        # 3. KMeans CLUSTERING in EMBEDDING SPACE
        # Minimises intra-cluster variance; groups semantically related findings.
        n_clusters = min(4, len(chunks))
        labels = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit_predict(embeddings)
        clusters: dict = {}
        for i, label in enumerate(labels.tolist()):
            clusters.setdefault(f"theme_{label}", []).append(chunks[i][:180] + "...")

        # 4. SENTIMENT ANALYSIS — DistilBERT SST-2
        # Binary {POSITIVE, NEGATIVE} + confidence score per chunk.
        results = [self.sentiment_pipe(c[:512])[0] for c in chunks]
        pos = sum(1 for r in results if r["label"] == "POSITIVE") / len(results)
        neg = sum(1 for r in results if r["label"] == "NEGATIVE") / len(results)
        sentiment = {
            "positive": round(pos, 2),
            "negative": round(neg, 2),
            "neutral":  round(max(0.0, 1.0 - pos - neg), 2),
        }

        # 5. NAMED ENTITY RECOGNITION — BERT CoNLL-2003
        # Extracts PER, ORG, LOC spans. Input sliced to 512 chars (BERT limit).
        entities: dict = {"ORG": set(), "PER": set(), "LOC": set()}
        for chunk in chunks[:6]:
            for ent in self.ner_pipe(chunk[:512]):
                grp = ent["entity_group"]
                if grp in entities:
                    entities[grp].add(ent["word"])

        print(
            f"  Sentiment : {sentiment['positive']*100:.0f}% pos / {sentiment['negative']*100:.0f}% neg  |"
            f"  Clusters : {len(clusters)} themes  |"
            f"  Entities : {sum(len(v) for v in entities.values())} extracted"
        )

        return MLInsights(
            sentiment=sentiment,
            clusters=clusters,
            entities={k: list(v)[:8] for k, v in entities.items()},
            top_findings=top_findings,
        )

print("MLProcessingLayer defined.")

MLProcessingLayer defined.


In [23]:
def research_agent(tasks: list[str]) -> list:
    """Search each task with DuckDuckGo, summarise with Mistral. Returns list[ResearchFinding]."""
    findings = []

    for task in tasks:
        print(f"  Searching: {task[:70]}...")

        # 1. Fetch raw snippets from DuckDuckGo (no API key required)
        with DDGS() as ddgs:
            results = list(ddgs.text(task, max_results=6))
        raw = "\n\n".join(
            f"{r['title']}: {r['body']}" for r in results if r.get("body")
        )

        if not raw:
            continue

        # 2. Ask Mistral to distil the snippets into a concise research summary
        summary = chat(
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a research analyst. "
                        "Summarise the web snippets below into 3-5 key facts, "
                        "statistics, and insights relevant to the search query."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Search query: {task}\n\nWeb snippets:\n{raw[:3000]}",
                },
            ],
            max_tokens=512,
        )

        if summary:
            findings.append(ResearchFinding(query=task, content=summary))

    return findings

print("research_agent defined.")

research_agent defined.


In [24]:
_ANALYSIS_TEMPLATE = """{
  "swot": {
    "strengths": ["strength 1", "strength 2", "strength 3"],
    "weaknesses": ["weakness 1", "weakness 2", "weakness 3"],
    "opportunities": ["opportunity 1", "opportunity 2", "opportunity 3"],
    "threats": ["threat 1", "threat 2", "threat 3"]
  },
  "key_metrics": ["metric 1", "metric 2", "metric 3", "metric 4"],
  "competitive_landscape": "description of competitive landscape",
  "main_insights": ["insight 1", "insight 2", "insight 3", "insight 4"]
}"""


def analyst_agent(query: str, findings: list, ml: MLInsights) -> Analysis:
    """Deep analysis via Mistral-7B. Returns a typed Analysis object."""

    research_text = "\n\n---\n\n".join(
        f"QUERY: {f.query}\n{f.content}" for f in findings
    )

    ml_context = (
        f"\nML INSIGHTS:\n"
        f"  Sentiment  — {ml.sentiment.get('positive', 0)*100:.0f}% positive, "
        f"{ml.sentiment.get('negative', 0)*100:.0f}% negative\n"
        f"  Clusters   — {len(ml.clusters)} themes (KMeans on MiniLM embeddings)\n"
        f"  Orgs (NER) — {', '.join(ml.entities.get('ORG', [])[:6])}\n"
        f"  People     — {', '.join(ml.entities.get('PER', [])[:5])}\n"
    )

    prompt = (
        f"You are a senior business analyst. Analyse: {query}\n\n"
        f"RESEARCH:\n{research_text[:5000]}\n\n"
        f"{ml_context}"
    )

    # 3000 tokens: ~500 reasoning preamble + ~800 JSON = safe margin
    data = chat_json(prompt, template=_ANALYSIS_TEMPLATE, max_tokens=3000)
    return Analysis(
        swot=SWOTAnalysis(**data["swot"]),
        key_metrics=data["key_metrics"],
        competitive_landscape=data["competitive_landscape"],
        main_insights=data["main_insights"],
    )

print("analyst_agent defined.")

analyst_agent defined.


In [25]:
def writer_agent(query: str, analysis: Analysis, ml: MLInsights) -> str:
    """Streams the final BI report token-by-token via HuggingFace Inference API."""

    swot = analysis.swot
    prompt = (
        f"Write a professional Business Intelligence Report for: **{query}**\n\n"
        "SWOT ANALYSIS:\n"
        "Strengths:\n"     + "\n".join(f"  - {s}" for s in swot.strengths)     + "\n"
        "Weaknesses:\n"    + "\n".join(f"  - {s}" for s in swot.weaknesses)    + "\n"
        "Opportunities:\n" + "\n".join(f"  - {s}" for s in swot.opportunities) + "\n"
        "Threats:\n"       + "\n".join(f"  - {s}" for s in swot.threats)       + "\n\n"
        "KEY METRICS:\n"   + "\n".join(f"  * {m}" for m in analysis.key_metrics) + "\n\n"
        f"COMPETITIVE LANDSCAPE:\n{analysis.competitive_landscape}\n\n"
        "ML-POWERED INSIGHTS:\n"
        f"  Sentiment: {ml.sentiment.get('positive', 0)*100:.0f}% positive, "
        f"{ml.sentiment.get('negative', 0)*100:.0f}% negative "
        f"(DistilBERT on {len(ml.top_findings)} ranked findings)\n"
        f"  {len(ml.clusters)} research themes auto-discovered via KMeans on "
        "sentence-transformer embeddings\n"
        f"  Key organisations (BERT NER): {', '.join(ml.entities.get('ORG', [])[:6])}\n\n"
        "Write the full report with these sections:\n"
        "1. Executive Summary\n"
        "2. ML-Powered Sentiment & Theme Analysis\n"
        "3. SWOT Analysis\n"
        "4. Competitive Landscape\n"
        "5. Key Metrics & Data Points\n"
        "6. Strategic Recommendations\n\n"
        "Be concise, data-driven, and professional."
    )

    print("\n" + "=" * 60)
    print("BUSINESS INTELLIGENCE REPORT")
    print("=" * 60 + "\n")

    full_report = ""
    stream = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are an expert business intelligence analyst. Write clear, data-driven reports.",
            },
            {"role": "user", "content": prompt},
        ],
        max_tokens=4096,
        temperature=0.3,
        stream=True,
    )
    for chunk in stream:
        token = chunk.choices[0].delta.content or ""
        print(token, end="", flush=True)
        full_report += token

    print("\n")
    return full_report

print("writer_agent defined.")

writer_agent defined.


In [26]:
def _plan_research(query: str) -> ResearchPlan:
    """Orchestrator: generate specific search queries as a numbered list."""
    prompt = (
        f"Create a research plan for: {query}\n\n"
        "Write 7 specific multi-word web search queries, numbered 1–7.\n"
        "Each query must be a full phrase (e.g. 'OpenAI revenue and valuation 2024'), "
        "NOT a single generic word like 'financials' or 'competitors'.\n"
        "Cover: revenue/valuation, main competitors, products/services, "
        "market share, recent news, funding, and regulatory challenges.\n\n"
        "1. [full search query about revenue/valuation]\n"
        "2. [full search query about main competitors]\n"
        "3. [full search query about products/services]\n"
        "4. [full search query about market share]\n"
        "5. [full search query about recent news]\n"
        "6. [full search query about funding]\n"
        "7. [full search query about regulation/challenges]"
    )
    raw = chat([{"role": "user", "content": prompt}], max_tokens=600, temperature=0.0)

    tasks = re.findall(r'^\s*\d+[\.\)]\s+(.+)$', raw, re.MULTILINE)
    # Drop any task that is a single word (the model echoing our topic headers)
    tasks = [t for t in tasks if len(t.split()) >= 3]
    if len(tasks) < 3:
        tasks = [l.strip() for l in raw.splitlines() if len(l.strip().split()) >= 3][:8]

    focus_areas = ["financials", "competitors", "products", "market trends"]
    return ResearchPlan(tasks=tasks[:8], focus_areas=focus_areas)


# Load ML models once — reused across all pipeline runs
print("Loading local ML models...")
_ml_layer = MLProcessingLayer()


def run_pipeline(query: str) -> str:
    """Full pipeline: plan → research → ML → analyse → write."""

    print(f"\n{'='*60}")
    print("MULTI-AGENT BI PIPELINE")
    print(f"Query: {query}")
    print(f"{'='*60}")

    # 1. ORCHESTRATOR
    print("\n[1/5] Orchestrator — planning research tasks...")
    plan = _plan_research(query)
    for t in plan.tasks:
        print(f"  · {t}")

    # 2. RESEARCH AGENT
    print(f"\n[2/5] Research Agent — searching {len(plan.tasks)} topics...")
    findings = research_agent(plan.tasks)
    print(f"  {len(findings)} findings gathered")

    # 3. ML LAYER
    print("\n[3/5] ML Layer — embeddings / clustering / sentiment / NER...")
    ml_insights = _ml_layer.process(query, findings)

    # 4. ANALYST AGENT
    print("\n[4/5] Analyst Agent — structured SWOT analysis...")
    analysis = analyst_agent(query, findings, ml_insights)
    print(f"  {len(analysis.main_insights)} key insights extracted")

    # 5. WRITER AGENT
    print("\n[5/5] Writer Agent — generating report...")
    return writer_agent(query, analysis, ml_insights)

print("Ready to run.")

Loading local ML models...
Loading local ML models...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8171.07it/s]


ML models ready.
Ready to run.


In [27]:
query = "Analyze OpenAI's competitive position in the AI industry"

report = run_pipeline(query)


MULTI-AGENT BI PIPELINE
Query: Analyze OpenAI's competitive position in the AI industry

[1/5] Orchestrator — planning research tasks...


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69c1b384-443f250c3ce4103c01887e3c;c4717357-5250-472b-8b0a-af72e4d197b0)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.